# Regression, and K-fold validation on a small dataset

506 samples, thirteen features on wildly different scales, and no validation set big enough to trust. K-fold is not a nicety here.

**Runs on:** CPU — about 3 minutes &nbsp;·&nbsp; **Slides:** [Chapter 4 — Classification and Regression](../../../course-web-slides/ch04/index.html) &nbsp;·&nbsp; **Section:** 03 — Predicting house prices

---

## A very small dataset

In [ ]:
from keras.datasets import boston_housing
import numpy as np

(train_data, train_targets), (test_data, test_targets) = \
    boston_housing.load_data()

print(train_data.shape, test_data.shape)
print("targets, in thousands of dollars:", train_targets[:6])
print("\nfeature ranges — note the scales:")
for i in range(train_data.shape[1]):
    col = train_data[:, i]
    print(f"  feature {i:2d}: {col.min():9.3f} .. {col.max():9.3f}")

Expected output:

```
(404, 13) (102, 13)
targets, in thousands of dollars: [15.2 42.3 50.  21.1 17.7 18.5]
```

## Normalization, and the rule about it

In [ ]:
mean = train_data.mean(axis=0)
std = train_data.std(axis=0)

train_data = (train_data - mean) / std
test_data = (test_data - mean) / std      # note: TRAIN statistics

print("after normalization, feature 0:",
      f"mean {train_data[:, 0].mean():+.3f}  std {train_data[:, 0].std():.3f}")

> ⚠️ **The test set is normalized with the **training** mean and standard deviation.** Using its own statistics leaks information from data you are pretending not to have — and the resulting number will be quietly optimistic.

## The model

In [ ]:
import keras
from keras import layers

def build_model():
    model = keras.Sequential([
        layers.Dense(64, activation="relu"),
        layers.Dense(64, activation="relu"),
        layers.Dense(1),                    # no activation
    ])
    model.compile(optimizer="rmsprop", loss="mse", metrics=["mae"])
    return model

**No activation on the last layer.** A sigmoid would cap the output at 1; anything else would constrain the range. Regression wants the layer left alone.

## K-fold, because 404 samples cannot spare a validation set

In [ ]:
k = 4
num_val_samples = len(train_data) // k
num_epochs = 100
all_scores = []

for i in range(k):
    print(f"fold {i}")
    val_data = train_data[i * num_val_samples: (i + 1) * num_val_samples]
    val_targets = train_targets[i * num_val_samples: (i + 1) * num_val_samples]
    partial_train_data = np.concatenate(
        [train_data[:i * num_val_samples],
         train_data[(i + 1) * num_val_samples:]], axis=0)
    partial_train_targets = np.concatenate(
        [train_targets[:i * num_val_samples],
         train_targets[(i + 1) * num_val_samples:]], axis=0)

    model = build_model()
    model.fit(partial_train_data, partial_train_targets,
              epochs=num_epochs, batch_size=16, verbose=0)
    _, val_mae = model.evaluate(val_data, val_targets, verbose=0)
    all_scores.append(val_mae)

print("\nper fold:", [round(s, 3) for s in all_scores])
print(f"mean {np.mean(all_scores):.3f}   spread {max(all_scores)-min(all_scores):.3f}")

Expected output:

```
per fold: [2.0xx, 2.9xx, 2.5xx, 2.4xx]
mean 2.5xx   spread 0.9xx
```

**The spread between folds is comparable to the differences you would be trying to measure.** That is the whole argument for K-fold: with a single split, which fold you happened to draw would decide your conclusion.

## Finding the right number of epochs

In [ ]:
import matplotlib.pyplot as plt

num_epochs = 200
all_mae_histories = []
for i in range(k):
    val_data = train_data[i * num_val_samples: (i + 1) * num_val_samples]
    val_targets_f = train_targets[i * num_val_samples: (i + 1) * num_val_samples]
    ptd = np.concatenate([train_data[:i * num_val_samples],
                          train_data[(i + 1) * num_val_samples:]], axis=0)
    ptt = np.concatenate([train_targets[:i * num_val_samples],
                          train_targets[(i + 1) * num_val_samples:]], axis=0)
    model = build_model()
    h = model.fit(ptd, ptt, validation_data=(val_data, val_targets_f),
                  epochs=num_epochs, batch_size=16, verbose=0)
    all_mae_histories.append(h.history["val_mae"])

average_mae = [np.mean([x[i] for x in all_mae_histories])
               for i in range(num_epochs)]

plt.figure(figsize=(7, 4.2))
plt.plot(range(11, len(average_mae) + 1), average_mae[10:], lw=1.5)
plt.xlabel("epoch"); plt.ylabel("validation MAE (averaged over 4 folds)")
plt.title(f"best around epoch {int(np.argmin(average_mae)) + 1}")
plt.show()

The first ten epochs are dropped so the rest is readable — their values are on a different scale entirely. Averaging over folds is what makes the minimum locatable at all; a single fold's curve is too noisy to read.

## The final model

In [ ]:
model = build_model()
model.fit(train_data, train_targets, epochs=130, batch_size=16, verbose=0)
_, test_mae = model.evaluate(test_data, test_targets, verbose=0)
print(f"test MAE: {test_mae:.3f}  (thousands of dollars)")

pred = model.predict(test_data, verbose=0).ravel()
plt.figure(figsize=(5, 5))
plt.scatter(test_targets, pred, s=14, alpha=.7)
lims = [0, max(test_targets.max(), pred.max()) + 3]
plt.plot(lims, lims, "k--", lw=1)
plt.xlabel("actual"); plt.ylabel("predicted"); plt.gca().set_aspect("equal")
plt.title("Predicted against actual"); plt.show()

Read the scatter, not the MAE. A model can hit a respectable average error while being systematically wrong at one end of the range — and here you can see whether it is.

---

## What to take away

- Regression: no activation on the last layer, `mse` loss, `mae` as the readable metric.
- Normalize with **training** statistics, always.
- On small data, K-fold is not a refinement — the fold-to-fold spread is as large as the effects you are measuring.
- Plot predicted against actual; an average error hides systematic bias.